In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
from tabnanny import verbose
import spikeinterface as si
import numpy as np
import scipy.spatial.distance
import pickle
import pandas as pd
from utils_clique import (
    process_unified_sorting,
    build_sliding_cliques,
    get_recording_clique
)

from scipy.io import loadmat

probe_data = loadmat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))


/home/ubuntu/.conda/envs/spike_sorting/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
recording_raw = se.read_intan(f"/home/ubuntu/Documents/jct/project/251205/M190011_260121_150111_merged_130.rhd", stream_id= '0', ignore_integrity_checks=True)

print('read success')

recording_raw = spre.unsigned_to_signed(recording_raw)
recording_raw = spre.resample(recording_raw, 10000)

recording_recorded = spre.bandpass_filter(recording_raw, freq_min=300, freq_max=3000)
recording_recorded = spre.notch_filter(recording_recorded, freq=50)
recording_f = spre.common_reference(recording_recorded, reference="global", operator="median")

recording_f = recording_f.set_probegroup(probe)

rec_params_raw = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
rec_params_raw = rec_params_raw[rec_params_raw['bhv_codes'] == 10]

original_fs = 30000
target_fs = 10000
fs_ratio = original_fs / target_fs
rec_params_raw['rec_codes_points_10000'] = (rec_params_raw['rec_codes_points'] / fs_ratio).astype(int)
rec_params_raw = rec_params_raw[(rec_params_raw['trial_ids'] >= 300) & (rec_params_raw['trial_ids'] < 5300)]
start_sample = rec_params_raw['rec_codes_points_10000'].iloc[0]
end_sample = rec_params_raw['rec_codes_points_10000'].iloc[-1]

recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)
recording_preprocessed = recording_segment.save(format="binary", n_jobs = 30)   


read success
Use cache_folder=/tmp/spikeinterface_cache/tmp7tou7mvw/PG232HSH
write_binary_recording 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=4.88 MiB - total_memory=146.48 MiB - chunk_duration=1.00s


write_binary_recording (workers: 30 processes): 100%|██████████| 3726/3726 [03:45<00:00, 16.53it/s]


In [3]:
output_folder = '/media/ubuntu/sda/visual_generation/results/neuroscroll_260122'
probe.set_contact_ids(recording_segment.channel_ids)

cliques = build_sliding_cliques(
    probe,
    clique_size=32,
    min_size=25,
    min_overlap=6,
    target_groups=10,
)

clique_info = {
    'cliques': cliques,  # List of CliqueInfo objects
    'clique_params': {
        'clique_size': 32,
        'min_size': 25,
        'min_overlap': 6,
        'target_groups': 10,
    },
    'probe_df': probe.to_dataframe(),  # Probe dataframe for verification
}

clique_info_path = f'{output_folder}/clique_info.pkl'
with open(clique_info_path, 'wb') as f:
    pickle.dump(clique_info, f)

[INFO] Built 10 cliques (target 10)
       Clique 00: channels 204-39 (32 channels)
       Clique 01: channels 74-81 (32 channels)
       Clique 02: channels 174-227 (32 channels)
       Clique 03: channels 33-88 (32 channels)
       Clique 04: channels 69-253 (32 channels)
       Clique 05: channels 229-232 (32 channels)
       Clique 06: channels 35-24 (32 channels)
       Clique 07: channels 207-141 (32 channels)
       Clique 08: channels 134-109 (32 channels)
       Clique 09: channels 148-115 (32 channels)


In [4]:
for clique in cliques:
    clique_id = clique.clique_id

    recording_clique = get_recording_clique(recording_preprocessed, clique)
    output_folder = f'/media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_{clique_id}'
    
    neuron_inf_all, gt_detect_array_all, analyzer = process_unified_sorting(
        recording_cmr=recording_clique,
        output_folder=output_folder ,
        distance_threshold=10.0,
        similarity_threshold=0.95,
        peak_sign='both',
        n_jobs=30,
        verbose=True
    )


开始统一处理整个recording的sorting结果

读取统一的sorting结果...
读取到 48 个units

创建analyzer并计算extensions...
estimate_sparsity 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


estimate_sparsity (workers: 30 processes): 100%|██████████| 3726/3726 [00:00<00:00, 8572.03it/s]

compute_waveforms 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s



compute_waveforms (workers: 30 processes): 100%|██████████| 3726/3726 [00:15<00:00, 248.09it/s]


完成extensions计算

发现 2 组需要merge的units，开始merge...
compute_waveforms 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=12.21 MiB - chunk_duration=1.00s


...

compute_waveforms 
engine=process - n_jobs=30 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=18.31 MiB - chunk_duration=1.00s


compute_waveforms (workers: 30 processes): 100%|██████████| 3726/3726 [00:10<00:00, 347.69it/s]


完成extensions计算

发现 1 组需要merge的units，开始merge...
compute_waveforms 
engine=process - n_jobs=20 - samples_per_chunk=10,000 - chunk_memory=625.00 KiB - total_memory=12.21 MiB - chunk_duration=1.00s


compute_waveforms (workers: 20 processes): 100%|██████████| 3726/3726 [00:15<00:00, 243.70it/s]


计算每个unit的channel_id...
完成channel_id计算，共处理48个units

计算channel_snr...
完成channel_snr计算，共处理48个units

生成整体的gt_detect_array...
保存neuron_inf_all和gt_detect_array_all...
已保存到: /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_9/neuron_inf_all.pickle 和 /media/ubuntu/sda/visual_generation/results/neuroscroll_260122/mountainsort/clique_9/gt_detect_array_all.csv

